# Lesson 5.3 - LLM Foundations & Prompt Engineering (HF demo)

This notebook demonstrates practical prompting patterns and LLM behavior using a small Hugging Face model when available.

## Objectives

- Run baseline prompt generation.
- Compare zero-shot vs few-shot behavior.
- Test constraint-oriented prompts (format/schema).
- Demonstrate chain-of-thought style prompting patterns (with safe summarized outputs).

## Setup

In [ ]:
from __future__ import annotations

import os
import random

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_ID = os.getenv("LLM_MODEL_ID", "distilgpt2")

model = None
tokenizer = None
load_error = None

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    model.to(device)
    model.eval()
except Exception as exc:
    load_error = str(exc)

{"model_loaded": model is not None, "model_id": MODEL_ID, "load_error": load_error}

In [ ]:
def generate(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8, top_p: float = 0.95) -> str:
    if model is None or tokenizer is None:
        return (
            "[stub-output] Model unavailable in this environment. "
            "Prompt received: " + prompt[:180]
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text

## Section A - Basic Generation

In [ ]:
prompt_basic = "Explain in simple terms what retrieval-augmented generation (RAG) is and why teams use it."
response_basic = generate(prompt_basic, max_new_tokens=120)
print("PROMPT:
", prompt_basic)
print("
MODEL OUTPUT:
", response_basic)

## Section B - Few-shot Prompt Example

In [ ]:
few_shot_prompt = (
    "Task: Convert user requests into concise action plans.\n\n"
    "Example 1\n"
    "Request: Build a weekly sales dashboard.\n"
    "Plan: Define metrics, ingest data, build aggregations, design visuals, validate with stakeholders.\n\n"
    "Example 2\n"
    "Request: Improve chatbot answer quality.\n"
    "Plan: Audit failures, improve retrieval, refine prompts, add evaluations, monitor regressions.\n\n"
    "Now do the same:\n"
    "Request: Launch an internal policy Q&A assistant.\n"
    "Plan:"
)

few_shot_output = generate(few_shot_prompt, max_new_tokens=100)
print(few_shot_output)

## Section C - Constraint/Format Prompt

In [ ]:
json_prompt = (
    "You are an assistant that MUST return valid JSON only.\n"
    "Create a response with keys: task, risks, mitigations.\n"
    "Task: Deploy a customer support AI assistant in healthcare."
)

json_like_output = generate(json_prompt, max_new_tokens=120, temperature=0.5)
print(json_like_output)

## Section D - Chain-of-Thought Style Prompt Template

In [ ]:
cot_style_prompt = (
    "Solve the following planning problem.\n"
    "First, think through key constraints privately.\n"
    "Then provide a concise final answer with exactly 4 bullet points.\n"
    "Problem: Our support bot hallucinates policy details. What should we do this quarter?"
)

cot_output = generate(cot_style_prompt, max_new_tokens=140, temperature=0.7)
print(cot_output)

## Connect to Theory

- Prompt structure (task + context + constraints + examples) directly affects output reliability.
- Few-shot prompting can reduce ambiguity without changing model weights.
- Constraint prompts often need post-validation (JSON schema parsing) in production.
- If repeated prompting fails to meet requirements, consider RAG and PEFT/fine-tuning.

## Business Case Studies & Exceptions

### Case 1: Customer Support Assistant

- **Pattern**: RAG-backed LLM drafts responses with citations from policy docs.
- **Benefit**: reduced handling time and consistent language.
- **Exception**: hallucinations in high-risk domains require guardrails + human review.

### Case 2: Internal Coding Assistant

- **Pattern**: prompt templates for refactor, tests, and docs generation with repository context.
- **Benefit**: faster engineering workflows.
- **Exception**: generated code must pass tests and security checks before merge.

### Case 3: When LLM Is Not the Right Tool

- Deterministic rule engines can outperform LLMs for strict compliance transformations.
- Ultra-low-latency systems may not tolerate model inference overhead.

## Interview Questions & Answers

1. **What is in-context learning?**  
   Adapting model behavior using prompt examples without updating model parameters.

2. **Why does prompt engineering matter?**  
   It is often the fastest, cheapest way to improve output quality and formatting.

3. **Difference between pre-training and fine-tuning?**  
   Pre-training learns broad language patterns; fine-tuning specializes behavior for tasks/domains.

4. **What is few-shot prompting?**  
   Providing a small set of task demonstrations in the prompt to guide output behavior.

5. **How do you reduce hallucinations?**  
   Use grounding via retrieval, explicit uncertainty instructions, and strict verification workflows.

6. **When should you fine-tune instead of prompt?**  
   When prompt-only approaches cannot meet consistent quality, latency, or cost targets.

7. **What is LoRA (high level)?**  
   Parameter-efficient fine-tuning using low-rank trainable updates on top of frozen base weights.

8. **How do you evaluate an LLM system?**  
   Offline benchmarks, human reviews, and production KPIs like groundedness, latency, and user outcomes.

9. **What is a good prompt template baseline?**  
   Role + objective + context + constraints + output schema + examples.

10. **Why are schema constraints important?**  
   Downstream systems need machine-parseable, predictable outputs.

11. **What is temperature?**  
   A sampling control that adjusts randomness and creativity versus determinism.

12. **How can long context hurt performance?**  
   Relevant signals can be diluted; retrieval and context compression become critical.